# Sumativa 2 — Fase 3: Núcleo Algorítmico, Eficiencia y POO

**Asignatura:** MCDI500 — Programación para la Ciencia de Datos  
**Dataset:** [lmarena-ai/arena-human-preference-140k](https://huggingface.co/datasets/lmarena-ai/arena-human-preference-140k)  
**Integrantes:** Alejandro Poirier, Angelo Morales, José Fredes, Valentina Carvajal  
**Fecha:** 15 de noviembre, 2026

---

## 1. Contexto y objetivo

En la Fase 2 se construyó un pipeline funcional que limpia y transforma el dataset LMArena (135 634 filas × 14 columnas originales). 
Dicho pipeline aplana columnas anidadas, convierte tipos y deriva variables de estilo para el análisis de preferencias entre respuestas de modelos de lenguaje.

**Objetivo de F3:** Refactorizar ese pipeline en clases reutilizables con programación orientada a objetos, 
verificar que el resultado sea idéntico al de F2, comparar la eficiencia entre implementaciones alternativas 
y documentar los patrones de diseño utilizados.


---

## 2. Importaciones y carga del dataset


In [1]:
import sys
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal

# Asegurar que src/ esté en el path
sys.path.insert(0, str(Path("..", "src").resolve()))

warnings.filterwarnings("ignore", category=FutureWarning)

A continuación se importan los módulos del paquete `src/arena/`:


In [2]:
from arena.io import PROJECT_ROOT, RAW_DIR, ensure_dataset
from arena.profile import load_raw
from arena.preprocessing import clean_dataset, derive_variables, flatten, convert_types
from arena.transformadores import (
    Transformador, recorrer,
    AplanadorConvMetadata, AplanadorCategoryTag,
    ConversorCategoria, ConversorTipo,
    DerivadorEstilo, EliminadorColumnas,
    pasos_f2, pasos_f3,
)
from arena.pipeline import Pipeline
from arena.medicion import medir, mejor_tiempo, comparar

### 2.1 Descarga y verificación de integridad

La función `ensure_dataset()` descarga los archivos parquet desde Hugging Face y verifica sus checksums SHA-256:


In [3]:
checksums = ensure_dataset()
for archivo, ok in checksums.items():
    print(f"  {archivo}: {'Verificación Correcta' if ok else 'Verificación Incorrecta'}")

  data/train-00000-of-00007.parquet: Verificación Correcta
  data/train-00001-of-00007.parquet: Verificación Correcta
  data/train-00002-of-00007.parquet: Verificación Correcta
  data/train-00003-of-00007.parquet: Verificación Correcta
  data/train-00004-of-00007.parquet: Verificación Correcta
  data/train-00005-of-00007.parquet: Verificación Correcta
  data/train-00006-of-00007.parquet: Verificación Correcta


### 2.2 Carga del dataset crudo

In [4]:
df_raw = load_raw()
print(f"Dataset cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
df_raw.head(2)

Dataset cargado: 135,634 filas × 11 columnas


,id,model_a,model_b,winner,evaluation_session_id,evaluation_order,conv_metadata,category_tag,language,is_code,timestamp
0,c4b9710c-8d64-4bee-a0b0-94637ae4cc65,gemini-2.5-pro,claude-3-7-sonnet-20250219-thinking-32k,model_a,a333a685-37f9-474d-b703-f079d8329552,1,"{'sum_assistant_a_tokens': 1854, 'header_count...",{'creative_writing_v0.1': {'creative_writing':...,en,False,2025-05-29 14:48:41.602925
1,7c44a466-fd07-4992-9764-176b75746a07,claude-3-5-haiku-20241022,claude-3-5-sonnet-20241022,tie,73e84b7b-d158-4d2d-9a7f-56286ec14564,1,"{'sum_assistant_a_tokens': 605, 'header_count_...",{'creative_writing_v0.1': {'creative_writing':...,pl,True,2025-07-07 02:04:51.977824


---

## 3. Pipeline F2 funcional (línea base)

Se ejecuta el pipeline funcional de F2 con `clean_dataset()`, que encadena: `load_raw()` → `flatten()` → `convert_types()` → `derive_variables()`. 

Este resultado sirve como referencia para verificar la equivalencia del pipeline POO.


In [5]:
df_f2 = clean_dataset()
print(f"F2 resultado: {df_f2.shape[0]:,} filas × {df_f2.shape[1]} columnas")
df_f2.dtypes

F2 resultado: 135,634 filas × 54 columnas


id                                            str
model_a                                       str
model_b                                       str
winner                                   category
evaluation_session_id                         str
evaluation_order                            int16
language                                 category
is_code                                      bool
timestamp                          datetime64[ns]
sum_assistant_a_tokens                      int64
header_a_h1                                 int64
header_a_h2                                 int64
header_a_h3                                 int64
header_a_h4                                 int64
header_a_h5                                 int64
header_a_h6                                 int64
list_a_ordered                              int64
list_a_unordered                            int64
bold_a_asterisk                             int64
bold_a_underscore                           int64


---

## 4. Pipeline F3 encapsulado en clases

### 4.1 Arquitectura de clases

El módulo `src/arena/transformadores.py` define una jerarquía de clases con el patrón **Template Method**:

- **`Transformador`** (clase base): define la secuencia `ajustar()` → `transformar()`. 
Controla el estado interno (`_ajustado`, `_parametros`) y delega la lógica concreta a dos métodos: `aprender()` y `aplicar()`.
- **Clases hijas**: cada una implementa `aprender()` (qué extraer del DataFrame de entrenamiento) y `aplicar()` (cómo transformar una copia del DataFrame).

| Clase | Columna | Operación |
|---|---|---|
| `AplanadorConvMetadata` | `conv_metadata` | Aplana diccionarios anidados de metadatos a 26 columnas |
| `AplanadorCategoryTag` | `category_tag` | Aplana etiquetas del clasificador a 12 columnas |
| `ConversorCategoria` | `winner`, `language` | Convierte texto a tipo categórico con vocabulario fijo |
| `ConversorTipo` | `is_code`, `evaluation_order` | Convierte a tipo fijo (`bool`, `int16`) |
| `DerivadorEstilo` | `winner` | Crea 7 variables derivadas de formato y tokens |
| `EliminadorColumnas` | `cat_creative_writing_score` | Elimina columnas redundantes |


### 4.2 Encapsulamiento

Cada transformador protege su estado mediante atributos con prefijo `_`:

- `_parametros`: diccionario con lo aprendido durante `ajustar()`. Solo se accede a través de la propiedad `parametros`, que devuelve una copia defensiva.
- `_ajustado`: flag que impide llamar a `transformar()` antes de `ajustar()`.

```python
@property
def parametros(self):
    return dict(self._parametros)  # copia defensiva
```

De esta forma, el estado interno no puede modificarse desde fuera de la clase.


### 4.3 Herencia y polimorfismo

`Transformador` define la interfaz común. El polimorfismo aparece cuando el `Pipeline` recorre su lista de transformadores y llama a `ajustar_transformar()` sobre cada uno, independientemente de su tipo concreto:

```python
for paso in self._pasos:
    intermedio = paso.ajustar_transformar(intermedio)
```

Cada clase hija responde de forma distinta al mismo mensaje (`aprender`, `aplicar`), lo que permite incorporar transformadores nuevos sin modificar el pipeline.


### 4.4 Construcción y ejecución del pipeline F3

In [6]:
# Crear el pipeline con los 9 pasos de F3
pipeline_f3 = Pipeline(pasos_f3())
print(f"Pipeline creado: {pipeline_f3}")
print(f"Cantidad de pasos: {len(pipeline_f3)}")
print()
pipeline_f3.resumen()

Pipeline creado: Pipeline(9 pasos, sin ajustar)
Cantidad de pasos: 9



,orden,paso,clase
0,1,AplanadorConvMetadata(conv_metadata),AplanadorConvMetadata
1,2,AplanadorCategoryTag(category_tag),AplanadorCategoryTag
2,3,ConversorCategoria(winner),ConversorCategoria
3,4,ConversorCategoria(language),ConversorCategoria
4,5,ConversorTipo(is_code),ConversorTipo
5,6,ConversorTipo(evaluation_order),ConversorTipo
6,7,DerivadorEstilo(winner),DerivadorEstilo
7,8,EliminadorColumnas(cat_creative_writing_score),EliminadorColumnas
8,9,ConversorTipo(cat_if_score),ConversorTipo


In [7]:
# Ajustar y transformar
pipeline_f3.ajustar(df_raw)
df_f3 = pipeline_f3.transformar(df_raw)
print(f"F3 resultado: {df_f3.shape[0]:,} filas × {df_f3.shape[1]} columnas")
df_f3.head(2)

F3 resultado: 135,634 filas × 53 columnas


,id,model_a,model_b,winner,evaluation_session_id,evaluation_order,language,is_code,timestamp,sum_assistant_a_tokens,...,cat_if,cat_if_score,cat_math,format_a,format_b,tokens_log_a,tokens_log_b,longer_response,more_formatted,winner_binary
0,c4b9710c-8d64-4bee-a0b0-94637ae4cc65,gemini-2.5-pro,claude-3-7-sonnet-20250219-thinking-32k,model_a,a333a685-37f9-474d-b703-f079d8329552,1,en,False,2025-05-29 14:48:41.602925,1854,...,False,1,False,112,6,7.52564,5.579730,a,a,a
1,7c44a466-fd07-4992-9764-176b75746a07,claude-3-5-haiku-20241022,claude-3-5-sonnet-20241022,tie,73e84b7b-d158-4d2d-9a7f-56286ec14564,1,pl,True,2025-07-07 02:04:51.977824,605,...,True,4,False,26,4,6.40688,6.011267,a,a,other


### 4.5 Parámetros aprendidos

Gracias al encapsulamiento, cada paso registra lo que extrajo del DataFrame de entrenamiento. A continuación se imprimen los parámetros de cada transformador:


In [8]:
for paso in pipeline_f3.pasos_ejecutados():
    params = paso.parametros
    resumen = {}
    for k, v in params.items():
        if isinstance(v, list) and len(v) > 5:
            resumen[k] = f"[{v[0]}, {v[1]}, ..., {v[-1]}] ({len(v)} elementos)"
        else:
            resumen[k] = v
    print(f"{paso.nombre}: {resumen}")

AplanadorConvMetadata(conv_metadata): {'columnas': '[sum_assistant_a_tokens, header_a_h1, ..., turns] (26 elementos)'}
AplanadorCategoryTag(category_tag): {'columnas': '[cat_creative_writing, cat_creative_writing_score, ..., cat_math] (12 elementos)'}
ConversorCategoria(winner): {'categorias': ['both_bad', 'model_a', 'model_b', 'tie']}
ConversorCategoria(language): {'categorias': '[<err>, aa, ..., zzp] (126 elementos)'}
ConversorTipo(is_code): {'tipo_original': 'bool', 'tipo_nuevo': 'bool'}
ConversorTipo(evaluation_order): {'tipo_original': 'int32', 'tipo_nuevo': 'int16'}
DerivadorEstilo(winner): {'columnas_format_a': '[header_a_h1, header_a_h2, ..., bold_a_underscore] (10 elementos)', 'columnas_format_b': '[header_b_h1, header_b_h2, ..., bold_b_underscore] (10 elementos)'}
EliminadorColumnas(cat_creative_writing_score): {'existe': True}
ConversorTipo(cat_if_score): {'tipo_original': 'float64', 'tipo_nuevo': 'Int8'}


---

## 5. Verificación de equivalencia F2 = F3

Se comprueba que el pipeline OOP produce exactamente el mismo resultado que la versión funcional de F2.

Los primeros 7 pasos de F3 replican la lógica de F2. 
Los pasos adicionales (eliminar columna redundante, convertir tipo) son correcciones propias de F3.


### 5.1 Equivalencia de los 7 pasos comunes

Se ejecuta un pipeline con solo los pasos de F2 (`pasos_f2()`) y se compara contra `clean_dataset()`:


In [9]:
# Pipeline OOP con solo los pasos de F2
pipeline_f2_oop = Pipeline(pasos_f2())
pipeline_f2_oop.ajustar(df_raw)
df_f2_oop = pipeline_f2_oop.transformar(df_raw)

# Ordenar columnas igual para comparar
columnas_comunes = sorted(set(df_f2.columns) & set(df_f2_oop.columns))
df_f2_sorted = df_f2[columnas_comunes].reset_index(drop=True)
df_f2_oop_sorted = df_f2_oop[columnas_comunes].reset_index(drop=True)

try:
    assert_frame_equal(df_f2_sorted, df_f2_oop_sorted, check_dtype=False, check_categorical=False)
    print("OK -->  Equivalencia verificada: el pipeline POO (pasos F2) produce el mismo resultado que clean_dataset().")
    print(f"Filas comparadas: {len(df_f2_sorted):,}")
    print(f"  Columnas comparadas: {len(columnas_comunes)}")
except AssertionError as e:
    print(f"NOT OK --> Diferencia encontrada:\n{e}")

OK -->  Equivalencia verificada: el pipeline POO (pasos F2) produce el mismo resultado que clean_dataset().
Filas comparadas: 135,634
  Columnas comparadas: 54


In [10]:
assert_frame_equal(
    df_f2.sort_index(axis=1),
    df_f2_oop.sort_index(axis=1),
    check_dtype=True,
    check_categorical=True,
    check_exact=True,
)
print("F2 funcional y F2 en clases coinciden en columnas, tipos y valores.")

F2 funcional y F2 en clases coinciden en columnas, tipos y valores.


### 5.2 Diferencias introducidas por F3

F3 agrega dos pasos respecto a F2:
1. `EliminadorColumnas("cat_creative_writing_score")`: elimina una columna redundante con `cat_creative_writing`.
2. `ConversorTipo("cat_if_score", "Int8")`: convierte a entero nullable para reducir uso de memoria.

Se verifica que las diferencias sean exactamente esas:


In [11]:
# Columnas presentes en F2 pero no en F3
cols_eliminadas = set(df_f2_oop.columns) - set(df_f3.columns)
print(f"Columnas eliminadas en F3: {cols_eliminadas}")

# Cambio de tipo en cat_if_score
tipo_f2 = df_f2_oop["cat_if_score"].dtype
tipo_f3 = df_f3["cat_if_score"].dtype
print(f"cambio de tipo en cat_if_score: era de tipo {tipo_f2} y cambia a {tipo_f3}")

# Comprobar que F3 solo eliminó la columna prevista.
eliminada = "cat_creative_writing_score"
convertida = "cat_if_score"

assert eliminada in df_f2_oop.columns
assert set(df_f3.columns) == set(df_f2_oop.columns) - {eliminada}

# Comprobar el cambio de tipo y que se conservaron los valores y faltantes.
assert str(df_f2_oop[convertida].dtype) == "float64"
assert str(df_f3[convertida].dtype) == "Int8"

assert_frame_equal(
    df_f2_oop[[convertida]],
    df_f3[[convertida]].astype("float64"),
    check_exact=True,
)

# Comparar todas las demás columnas, incluidos sus tipos y categorías.
resto_f2 = df_f2_oop.drop(columns=[eliminada, convertida]).sort_index(axis=1)
resto_f3 = df_f3.drop(columns=[convertida]).sort_index(axis=1)

assert_frame_equal(
    resto_f2,
    resto_f3,
    check_dtype=True,
    check_categorical=True,
    check_exact=True,
)

print(f"Verificación estricta aprobada en {len(resto_f3):,} filas.")
print(f"Las {resto_f3.shape[1]} columnas restantes coinciden en valores y tipos.")

Columnas eliminadas en F3: {'cat_creative_writing_score'}
cambio de tipo en cat_if_score: era de tipo float64 y cambia a Int8
Verificación estricta aprobada en 135,634 filas.
Las 52 columnas restantes coinciden en valores y tipos.


In [12]:
pd.DataFrame([
    {
        "Columna": "cat_creative_writing_score",
        "Tipo en F2": str(df_f2_oop["cat_creative_writing_score"].dtype),
        "Tipo en F3": "Columna eliminada",
        "Motivo": "Se eliminó por redundancia con cat_creative_writing.",
    },
    {
        "Columna": "cat_if_score",
        "Tipo en F2": str(df_f2_oop["cat_if_score"].dtype),
        "Tipo en F3": str(df_f3["cat_if_score"].dtype),
        "Motivo": "Se usó un entero que admite valores faltantes para reducir el uso de memoria.",
    },
])

,Columna,Tipo en F2,Tipo en F3,Motivo
0,cat_creative_writing_score,str,Columna eliminada,Se eliminó por redundancia con cat_creative_wr...
1,cat_if_score,float64,Int8,Se usó un entero que admite valores faltantes ...


---

## 6. Medición de eficiencia: bucle vs. vectorizado

Se comparan dos implementaciones de la operación `derive_variables` para cuantificar 
la diferencia de rendimiento entre un bucle fila por fila y operaciones vectorizadas con NumPy/pandas.


In [13]:
def derivar_con_bucle(df):
    """Versión con bucle: calcula las 7 variables derivadas fila por fila."""
    df = df.copy()
    header_a = [c for c in df.columns if c.startswith("header_a_")]
    list_a   = [c for c in df.columns if c.startswith("list_a_")]
    bold_a   = [c for c in df.columns if c.startswith("bold_a_")]
    header_b = [c for c in df.columns if c.startswith("header_b_")]
    list_b   = [c for c in df.columns if c.startswith("list_b_")]
    bold_b   = [c for c in df.columns if c.startswith("bold_b_")]
    cols_format_a = header_a + list_a + bold_a
    cols_format_b = header_b + list_b + bold_b
    mapa_winner = {"model_a": "a", "model_b": "b", "tie": "other", "both_bad": "other"}
    n = len(df)
    format_a_vals, format_b_vals = [0]*n, [0]*n
    tokens_log_a_vals, tokens_log_b_vals = [0.0]*n, [0.0]*n
    longer_vals, more_fmt_vals, winner_bin_vals = [""]*n, [""]*n, [""]*n

    for i in range(n):
        row = df.iloc[i]
        fa = sum(int(row[c]) for c in cols_format_a)
        fb = sum(int(row[c]) for c in cols_format_b)
        format_a_vals[i], format_b_vals[i] = fa, fb
        tokens_log_a_vals[i] = math.log1p(row["sum_assistant_a_tokens"])
        tokens_log_b_vals[i] = math.log1p(row["sum_assistant_b_tokens"])
        longer_vals[i] = "a" if row["sum_assistant_a_tokens"] > row["sum_assistant_b_tokens"] else ("b" if row["sum_assistant_a_tokens"] < row["sum_assistant_b_tokens"] else "equal")
        more_fmt_vals[i] = "a" if fa > fb else ("b" if fa < fb else "equal")
        winner_bin_vals[i] = mapa_winner.get(row["winner"], "other")

    df["format_a"], df["format_b"] = format_a_vals, format_b_vals
    df["tokens_log_a"] = pd.array(tokens_log_a_vals, dtype="float32")
    df["tokens_log_b"] = pd.array(tokens_log_b_vals, dtype="float32")
    df["longer_response"], df["more_formatted"] = longer_vals, more_fmt_vals
    df["winner_binary"] = winner_bin_vals
    return df


In [14]:
# Preparar datos pre-derivación
df_pre = convert_types(flatten(df_raw))

# Verificar equivalencia de ambas implementaciones
df_vec  = derive_variables(df_pre)
df_loop = derivar_con_bucle(df_pre)
cols_derivadas = ["format_a", "format_b", "tokens_log_a", "tokens_log_b",
                  "longer_response", "more_formatted", "winner_binary"]
assert_frame_equal(
    df_vec[cols_derivadas].reset_index(drop=True),
    df_loop[cols_derivadas].reset_index(drop=True),
    check_dtype=False,
)
print("OK --> Ambas implementaciones producen el mismo resultado")

# Medir sobre muestra de 20,000 filas
muestra = df_pre.sample(n=20_000, random_state=42).reset_index(drop=True)
tabla = comparar(
    {"bucle": derivar_con_bucle, "vectorizado": derive_variables},
    muestra, repeticiones=3,
)
print(f"\nComparación sobre {len(muestra):,} filas:")
tabla


OK --> Ambas implementaciones producen el mismo resultado

Comparación sobre 20,000 filas:


,implementacion,segundos,memoria_mb,veces_mas_lenta
0,bucle,2.928572,8.160446,199.860234
1,vectorizado,0.014653,5.827110,1.000000


La versión vectorizada delega el cómputo a rutinas compiladas en C (NumPy/pandas) que procesan arreglos completos, 
mientras que el bucle recorre cada fila con `iloc[i]`, incurriendo en el overhead del intérprete de Python por cada iteración. 
La diferencia se amplifica conforme crece el número de filas.


---

## 7. Recursividad: la función `recorrer()`

### 7.1 Definición

`recorrer(valor, ruta=())` es un generador recursivo que aplana un diccionario anidado a cualquier profundidad, produciendo pares `(ruta, hoja)`:

```python
def recorrer(valor, ruta=()):
    if not isinstance(valor, dict):     # caso base: hoja
        yield ruta, valor
        return
    for clave, interior in valor.items():  # caso recursivo
        yield from recorrer(interior, ruta + (str(clave),))
```

- **Caso base**: cuando `valor` no es un diccionario, se entrega la ruta acumulada junto con el valor.
- **Caso recursivo**: se itera sobre las claves y se desciende un nivel, acumulando la clave actual en la ruta.

`yield from` propaga los resultados del generador anidado sin necesidad de un bucle adicional.


### 7.2 Ejemplo de ejecución

In [15]:
# Ejemplo con un diccionario anidado similar a conv_metadata
ejemplo = {
    "header_count_a": {"h1": 0, "h2": 3, "h3": 1},
    "sum_user_tokens": 42,
    "bold_count_b": {"**": 5, "__": 0},
}

print("Entradas del generador recorrer():")
for ruta, hoja in recorrer(ejemplo):
    print(f"  ruta={ruta}, hoja={hoja}")

Entradas del generador recorrer():
  ruta=('header_count_a', 'h1'), hoja=0
  ruta=('header_count_a', 'h2'), hoja=3
  ruta=('header_count_a', 'h3'), hoja=1
  ruta=('sum_user_tokens',), hoja=42
  ruta=('bold_count_b', '**'), hoja=5
  ruta=('bold_count_b', '__'), hoja=0


### 7.3 Uso en los transformadores

`AplanadorAnidado._filas_planas()` usa `recorrer()` para convertir cada diccionario de una columna del DataFrame en un diccionario plano, 
donde las claves se construyen concatenando las rutas.

Las clases hijas (`AplanadorConvMetadata`, `AplanadorCategoryTag`) implementan `nombre_columna(ruta)` para definir el nombre de cada columna resultante.


In [16]:
# Demostración: aplanar una fila de conv_metadata
fila_ejemplo = df_raw["conv_metadata"].iloc[0]
aplanador = AplanadorConvMetadata()

# Mostrar las primeras columnas generadas
filas_planas = aplanador._filas_planas(df_raw["conv_metadata"].head(1))
for nombre, valor in list(filas_planas[0].items())[:8]:
    print(f"  {nombre}: {valor}")

  sum_assistant_a_tokens: 1854
  header_a_h1: 0
  header_a_h2: 0
  header_a_h3: 8
  header_a_h4: 0
  header_a_h5: 0
  header_a_h6: 0
  list_a_ordered: 0


---

## 8. Validación técnica

La suite de pruebas del proyecto se encuentra en `tests/` y se ejecuta con `pytest`. 
Contiene pruebas unitarias para transformadores, pipeline, carga de datos y medición de eficiencia.

A continuación se muestran las validaciones principales realizadas en este notebook:


In [17]:
# Validación 1: el pipeline OOP reproduce exactamente el resultado funcional de F2
print(f"F2: las versiones funcional y en clases coinciden en las {df_f2.shape[1]} columnas.")
print(f"F3: se verificaron los dos cambios documentados en la sección 5.2. "
      f"Las otras {resto_f3.shape[1]} columnas conservan sus valores y tipos "
      f"en las {len(resto_f3)} filas.")

# Validación 2: ambas implementaciones de eficiencia producen el mismo resultado
print(f"Equivalencia bucle=vectorizado verificada sobre 7 columnas derivadas")

# Validación 3: transformar sin ajustar lanza RuntimeError
try:
    ConversorCategoria("winner").transformar(df_raw)
    print("NOT OK --> Se esperaba RuntimeError")
except RuntimeError as e:
    print(f"OK --> RuntimeError al transformar sin ajustar: {e}")

# Validación 4: agregar un no-transformador al Pipeline lanza TypeError
try:
    Pipeline().agregar("no soy un transformador")
    print("NOT OK --> Se esperaba TypeError")
except TypeError as e:
    print(f"OK --> TypeError al agregar objeto inválido: {e}")


F2: las versiones funcional y en clases coinciden en las 54 columnas.
F3: se verificaron los dos cambios documentados en la sección 5.2. Las otras 52 columnas conservan sus valores y tipos en las 135634 filas.
Equivalencia bucle=vectorizado verificada sobre 7 columnas derivadas
OK --> RuntimeError al transformar sin ajustar: ConversorCategoria(winner): hay que ajustar antes de transformar.
OK --> TypeError al agregar objeto inválido: Se esperaba un Transformador y se recibió str.


---

## 9. Patrones de diseño

### 9.1 Template Method

Patrón central del proyecto. `Transformador` define la secuencia del algoritmo en `ajustar()` y `transformar()`, 
mientras que las subclases concretan los pasos en `aprender()` y `aplicar()`.

**Estructura:**

```
Transformador (clase base)
├── ajustar(df)        → valida columna, llama a aprender(), marca _ajustado
│   └── aprender(df)   → [abstracto] cada hija lo implementa
├── transformar(df)    → verifica _ajustado, llama a aplicar() sobre copia
│   └── aplicar(df)    → [abstracto] cada hija lo implementa
└── ajustar_transformar(df) → atajo que combina ambos
```

Se eligió este patrón porque el flujo de un transformador siempre sigue la misma secuencia 
(validar → aprender → marcar estado → aplicar), pero la lógica concreta cambia según la operación: 
aplanar un diccionario anidado no tiene relación con convertir un tipo de dato. 
El patrón permite escribir la validación y el control de estado una sola vez en la clase base, 
mientras que cada hija se ocupa únicamente de su operación.

Gamma et al. (1994) lo describen como una forma de "definir el esqueleto de un algoritmo en una operación, 
delegando algunos pasos a las subclases".

**Ventajas concretas en este proyecto:**
- La validación y el control de estado se escriben una vez en la base.
- Agregar un transformador nuevo requiere solo implementar `aprender()` y `aplicar()`.
- El pipeline trata a todos los transformadores de forma uniforme gracias al polimorfismo.

### 9.2 Pipeline (encadenamiento)

El pipeline encadena transformadores y los ejecuta en orden. Cada paso recibe el DataFrame que devolvió el anterior:

```python
for paso in self._pasos:
    intermedio = paso.ajustar_transformar(intermedio)
```

Esto permite armar comportamiento complejo a partir de piezas simples, 
donde cada transformador tiene:
- alta cohesión (hace una sola cosa)
- bajo acoplamiento (no conoce a los demás).


### 9.3 Cohesión y acoplamiento

**Alta cohesión:** cada transformador tiene una sola responsabilidad. 
- `AplanadorConvMetadata` solo aplana metadatos
- `ConversorCategoria` solo convierte una columna a tipo categórico. 

Los parámetros aprendidos son específicos de cada tarea.

**Bajo acoplamiento:** los transformadores no se referencian entre sí. 
El pipeline los recorre en secuencia y el único contrato es la interfaz `ajustar/transformar` de la clase base. 
Se puede agregar, quitar o reordenar pasos sin modificar el resto.

La excepción parcial es `DerivadorEstilo`, que depende de que las columnas de formato ya existan 
(requiere que `AplanadorConvMetadata` se ejecute antes). 
Esta dependencia de orden es inherente al dominio, no al diseño de las clases.


---

## 10. Conclusiones y proyección F4

### Resumen de F3

1. **POO aplicada**: se encapsuló el pipeline de limpieza en una jerarquía de clases con herencia, polimorfismo y encapsulamiento. Se verificó equivalencia con F2.
2. **Eficiencia medida**: la comparación entre un bucle fila por fila y operaciones vectorizadas confirma la ventaja de NumPy/pandas, con diferencias que crecen al aumentar el volumen de datos.
3. **Validación técnica**: se probaron casos normales, límite y de excepción sobre transformadores y pipeline.
4. **Recursividad**: `recorrer()` resuelve el aplanamiento de diccionarios anidados de forma genérica, sin depender de la profundidad.
5. **Patrones documentados**: Template Method como patrón central y Pipeline como patrón de composición.

### Proyección F4

En la Fase 4 se trabajará con análisis estadístico y visualización de los datos preprocesados, 
explorando las preferencias entre modelos de lenguaje y la influencia del formato de respuesta en las decisiones.


---

## Referencias

- Chiang, W.-L., Zheng, L., Sheng, Y., et al. (2024). Chatbot Arena: An Open Platform for Evaluating LLMs by Human Preference. *arXiv preprint arXiv:2403.04132*. https://arxiv.org/abs/2403.04132
- Gamma, E., Helm, R., Johnson, R., & Vlissides, J. (1994). *Design Patterns: Elements of Reusable Object-Oriented Software*. Addison-Wesley.
- magistercienciadatos. (2026). *MCDI500 · Material didáctico de la Semana 1* [Repositorio de materiales del curso]. GitHub. https://github.com/magistercienciadatos/Material_Didactico_Semana1
- NumPy development team. (2024). *NumPy documentation — numpy.log1p*. https://numpy.org/doc/stable/reference/generated/numpy.log1p.html
- pandas development team. (2024). *pandas documentation — CategoricalDtype*. https://pandas.pydata.org/docs/reference/api/pandas.CategoricalDtype.html
- Python Software Foundation. (2024). *The Python Language Reference — Classes*. https://docs.python.org/3/reference/datamodel.html
- Salinas Silva, O. (2026). *Apuntes técnicos Semana 1: Estructura de datos, funciones, módulos, manejo de excepciones, NumPy y Pandas* [Material docente]. MCDI500 Programación para la Ciencia de Datos, Universidad Andrés Bello.
- *Sumativa 2. Avance del proyecto fase 3 – Núcleo algorítmico, eficiencia e implementación orientada a objetos.* (2026). Enunciado y rúbrica de MCDI500, Universidad Andrés Bello (Canvas).
